# Static VAR compensator (SVC) voltage control

DPsim's Newton-Raphson powerflow can hold an `SP::Ph1::SVC` bus at a voltage setpoint by moving the device's reactive injection within its Q band, an outer loop that runs alongside the generator Q-limit switching. Enforcement is opt-in via `set_pf_solver_enforce_svc_control(True)` on the `Simulation` (default off). Inside the band the SVC holds the bus at its setpoint; at the band edge the injection pins at Qmax or Qmin and the voltage misses the setpoint.

This notebook uses a hand-wired three-bus case (slack, two PiLines, a mid-bus load, and an SVC at the far bus) and checks the behaviour under both the dense and sparse solvers.

In [ ]:
import math

import numpy as np
import pandas as pd

import dpsimpy

In [ ]:
BASE_V = 110e3
BASE_S = 100e6


def run_pf(
    v_set_pu, q_max_mvar, q_min_mvar, load_p_mw, load_q_mvar, enforce, use_sparse
):
    n1 = dpsimpy.sp.SimNode("n1", dpsimpy.PhaseType.Single)
    n2 = dpsimpy.sp.SimNode("n2", dpsimpy.PhaseType.Single)
    n3 = dpsimpy.sp.SimNode("n3", dpsimpy.PhaseType.Single)

    slack = dpsimpy.sp.ph1.SynchronGenerator("slack")
    slack.set_parameters(
        rated_apparent_power=BASE_S,
        rated_voltage=BASE_V,
        set_point_active_power=0.0,
        set_point_voltage=BASE_V,
        powerflow_bus_type=dpsimpy.PowerflowBusType.VD,
    )
    slack.set_base_voltage(BASE_V)
    slack.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.VD)
    slack.connect([n1])

    load = dpsimpy.sp.ph1.Load("load")
    load.set_parameters(load_p_mw * 1e6, load_q_mvar * 1e6, BASE_V)
    load.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.PQ)
    load.connect([n2])

    svc = dpsimpy.sp.ph1.SVC("svc")
    svc.set_parameters(
        rated_apparent_power=BASE_S,
        rated_voltage=BASE_V,
        set_point_voltage=v_set_pu * BASE_V,
        q_limit_max=q_max_mvar * 1e6,
        q_limit_min=q_min_mvar * 1e6,
    )
    svc.set_base_voltage(BASE_V)
    svc.connect([n3])

    z_base = BASE_V**2 / BASE_S
    line12 = dpsimpy.sp.ph1.PiLine("line12")
    line12.set_parameters(0.02 * z_base, 0.08 * z_base / (2 * math.pi * 50), 0.0, 0.0)
    line12.set_base_voltage(BASE_V)
    line12.connect([n1, n2])

    line23 = dpsimpy.sp.ph1.PiLine("line23")
    line23.set_parameters(0.02 * z_base, 0.08 * z_base / (2 * math.pi * 50), 0.0, 0.0)
    line23.set_base_voltage(BASE_V)
    line23.connect([n2, n3])

    system = dpsimpy.SystemTopology(
        50, [n1, n2, n3], [slack, load, svc, line12, line23]
    )

    sim = dpsimpy.Simulation("svc_pf", dpsimpy.LogLevel.off)
    sim.set_system(system)
    sim.set_time_step(0.1)
    sim.set_final_time(0.1)
    sim.set_domain(dpsimpy.Domain.SP)
    sim.set_solver(dpsimpy.Solver.NRP)
    sim.set_solver_component_behaviour(dpsimpy.SolverBehaviour.Simulation)
    sim.set_pf_solver_use_sparse(use_sparse)
    sim.set_pf_solver_enforce_svc_control(enforce)
    sim.run()

    v_svc = abs(complex(n3.single_voltage()) / BASE_V)
    q_svc = svc.get_apparent_power().imag / 1e6
    return v_svc, q_svc

The load moves the far bus off nominal: an inductive load pulls it down, a capacitive load pushes it up. A wide Q band lets the SVC reach its setpoint; a tight band forces it to pin at the reactive limit and miss. With enforcement off the SVC injects nothing and the bus is left uncompensated. The cases below exercise a hold by injecting, a hold by absorbing, a pin at each limit, and the enforcement-off baseline.

In [ ]:
# label, Vset [pu], Qmax [MVAr], Qmin [MVAr], load P [MW], load Q [MVAr], enforce, expected
CASES = [
    ("wide band, inductive load, holds", 1.00, 1000, -1000, 30, 15, True, "hold"),
    ("tight Qmax, inductive load, pins max", 1.00, 5, -5, 30, 15, True, "max"),
    ("wide band, capacitive load, holds", 0.98, 1000, -1000, 5, -60, True, "hold"),
    ("tight Qmin, capacitive load, pins min", 0.98, 5, -5, 5, -60, True, "min"),
    ("enforcement off, no support", 1.00, 1000, -1000, 30, 15, False, "off"),
]

In [ ]:
rows = []
for use_sparse in (False, True):
    for label, v_set, q_max, q_min, load_p, load_q, enforce, expected in CASES:
        v, q = run_pf(v_set, q_max, q_min, load_p, load_q, enforce, use_sparse)
        rows.append(
            {
                "case": label,
                "solver": "sparse" if use_sparse else "dense",
                "Vset [pu]": v_set,
                "Qmax [MVAr]": q_max,
                "Qmin [MVAr]": q_min,
                "expected": expected,
                "V [pu]": v,
                "SVC Q [MVAr]": q,
            }
        )
results = pd.DataFrame(rows)
results

In [ ]:
for _, row in results.iterrows():
    if row["expected"] == "max":
        assert (
            abs(row["SVC Q [MVAr]"] - row["Qmax [MVAr]"]) < 0.5
        ), f"{row['solver']}/{row['case']}: SVC not pinned at Qmax"
    elif row["expected"] == "min":
        assert (
            abs(row["SVC Q [MVAr]"] - row["Qmin [MVAr]"]) < 0.5
        ), f"{row['solver']}/{row['case']}: SVC not pinned at Qmin"
    elif row["expected"] == "hold":
        assert (
            abs(row["V [pu]"] - row["Vset [pu]"]) < 1e-3
        ), f"{row['solver']}/{row['case']}: SVC bus not held at Vset"
    else:
        assert (
            abs(row["SVC Q [MVAr]"]) < 1e-6
        ), f"{row['solver']}/{row['case']}: SVC injected with enforcement off"

print("all SVC voltage-control cases passed")